# Lesson 4 · Where should we set the cutoff?
**Optional Python experiment · about 10–15 minutes**

In class, a cutoff turned a measurement into a positive or negative test. Here you'll explore a new question: **What do we gain and lose by moving that cutoff?**

## 1 · Predict
Imagine people with and without a condition have overlapping test measurements. A higher measurement makes the condition more likely, but there is no perfect dividing line.

**Before running the graph:** As we raise the cutoff, what do you predict happens to:
- **Sensitivity (recall):** the fraction of affected people identified?
- **Precision:** the fraction of positive tests that are true positives?

In [ ]:
# Run once: helper functions for the graph. No extra packages required.
import math
from IPython.display import HTML, display

def chance_above(cutoff, mean, spread=1):
    """Chance of a test measurement exceeding the cutoff."""
    return 0.5 * math.erfc((cutoff - mean) / (spread * math.sqrt(2)))

def curve_data(prevalence, separation):
    rows = []
    for i in range(121):
        cutoff = -1 + 5 * i / 120
        recall = chance_above(cutoff, separation)
        false_positive_rate = chance_above(cutoff, 0)
        positive_fraction = prevalence*recall + (1-prevalence)*false_positive_rate
        precision = prevalence*recall/positive_fraction if positive_fraction else 0
        rows.append((cutoff, recall, precision))
    return rows


def plot_histograms(prevalence, separation, cutoff, total_people=1000):
    """Expected histogram counts for the same hypothetical population used below."""
    def cdf(x, mean):
        return 0.5 * (1 + math.erf((x - mean) / math.sqrt(2)))
    bins = [-1 + 6*i/16 for i in range(17)]
    groups = []
    for mean, n, color, label in [
        (0, total_people*(1-prevalence), "#75b7c0", "Without condition"),
        (separation, total_people*prevalence, "#d58951", "With condition")
    ]:
        counts = [(cdf(b, mean)-cdf(a, mean))*n for a,b in zip(bins[:-1],bins[1:])]
        groups.append((counts, color, label))
    ymax = max(max(counts) for counts,_,_ in groups) * 1.1
    W,H,L,T,R,B = 750,300,72,38,22,48
    sx=lambda x: L+(x+1)/6*(W-L-R)
    sy=lambda count: H-B-(count/ymax)*(H-T-B)
    bits=[f'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 {W} {H}" role="img" aria-label="Overlapping histograms of test measurements">',
          f'<rect width="{W}" height="{H}" fill="white"/>']
    for fraction in (0,.25,.5,.75,1):
        count=fraction*ymax
        y=sy(count)
        bits += [f'<line x1="{L}" x2="{W-R}" y1="{y:.1f}" y2="{y:.1f}" stroke="#e5e8e9"/>',
                 f'<text x="{L-10}" y="{y+4:.1f}" text-anchor="end" font-size="12" fill="#555">{count:.0f}</text>']
    for counts,color,label in groups:
        for i,count in enumerate(counts):
            left=sx(bins[i])+1
            width=sx(bins[i+1])-sx(bins[i])-2
            bits.append(f'<rect x="{left:.1f}" y="{sy(count):.1f}" width="{width:.1f}" height="{sy(0)-sy(count):.1f}" fill="{color}" fill-opacity=".66"/>')
    x=sx(cutoff)
    bits += [f'<line x1="{x:.1f}" x2="{x:.1f}" y1="{T}" y2="{H-B}" stroke="#26323b" stroke-width="2" stroke-dasharray="6 5"/>',
             f'<text x="{min(x+8,W-85):.1f}" y="{T+15}" font-size="13" fill="#26323b">Cutoff</text>']
    for tick in range(-1,6):
        bits.append(f'<text x="{sx(tick):.1f}" y="{H-B+20}" text-anchor="middle" font-size="12" fill="#555">{tick}</text>')
    bits += [f'<rect x="{L}" y="7" width="14" height="14" fill="#75b7c0"/>',
             f'<text x="{L+20}" y="20" font-size="13">Without condition</text>',
             f'<rect x="{L+200}" y="7" width="14" height="14" fill="#d58951"/>',
             f'<text x="{L+220}" y="20" font-size="13">With condition</text>',
             f'<text x="{W/2}" y="{H-6}" font-size="14" text-anchor="middle">Test measurement (arbitrary units)</text>',
             f'<text x="16" y="{H/2}" font-size="13" text-anchor="middle" transform="rotate(-90 16 {H/2})">Expected number of people</text>',
             '</svg>']
    display(HTML(''.join(bits)))

def plot_tradeoff(rows, current_cutoff):
    W,H,L,T,R,B = 750,350,65,32,30,52
    pw,ph = W-L-R,H-T-B
    sx=lambda x: L+(x+1)/5*pw
    sy=lambda y: T+(1-y)*ph
    points=lambda idx: ' '.join(f'{sx(x):.1f},{sy(row[idx]):.1f}' for row in rows for x in [row[0]])
    current = min(rows,key=lambda r:abs(r[0]-current_cutoff))
    bits=[f'<svg viewBox="0 0 {W} {H}" xmlns="http://www.w3.org/2000/svg" role="img" aria-label="Sensitivity and precision versus test cutoff">',
          f'<rect width="{W}" height="{H}" fill="white"/>']
    for v in [0,.25,.5,.75,1]:
        bits.append(f'<line x1="{L}" y1="{sy(v)}" x2="{W-R}" y2="{sy(v)}" stroke="#e6e9eb"/>')
        bits.append(f'<text x="{L-12}" y="{sy(v)+5}" text-anchor="end" font-size="13" fill="#555">{v*100:.0f}%</text>')
    for x in [-1,0,1,2,3,4]:
        bits.append(f'<text x="{sx(x)}" y="{H-B+23}" text-anchor="middle" font-size="13" fill="#555">{x}</text>')
    bits += [f'<polyline points="{points(1)}" fill="none" stroke="#287e87" stroke-width="3.5"/>',
             f'<polyline points="{points(2)}" fill="none" stroke="#c86c32" stroke-width="3.5"/>',
             f'<line x1="{sx(current[0])}" y1="{T}" x2="{sx(current[0])}" y2="{H-B}" stroke="#333" stroke-width="1.7" stroke-dasharray="5 5"/>',
             f'<circle cx="{sx(current[0])}" cy="{sy(current[1])}" r="5" fill="#287e87"/>',
             f'<circle cx="{sx(current[0])}" cy="{sy(current[2])}" r="5" fill="#c86c32"/>',
             f'<text x="{W/2}" y="{H-8}" font-size="15" text-anchor="middle">Test cutoff (arbitrary measurement units)</text>',
             f'<text x="{L}" y="19" font-size="15" fill="#287e87">Sensitivity / recall</text>',
             f'<text x="{L+190}" y="19" font-size="15" fill="#c86c32">Precision</text>', '</svg>']
    display(HTML(''.join(bits)))
    print(f'At cutoff {current[0]:.2f}: sensitivity = {current[1]:.1%}, precision = {current[2]:.1%}')

## 2 · Move the cutoff
The model assumes unaffected people have measurements centered at **0**, and affected people at **2**, with overlapping spreads. This is an illustrative model, **not data from real patients**.

**Compare the graphs:** The cutoff line in the histograms represents the same cutoff marked on the sensitivity/precision curves below.

Change `CUTOFF`, run the cell again, and look for a point where gaining precision begins to cost substantial sensitivity.

In [ ]:
# Change these values, then rerun this cell.
PREVALENCE = 0.05   # 5% of people actually have the condition
SEPARATION = 2.0    # how far apart the two groups' measurements are
CUTOFF = 1.0        # TRY: 0.0, 1.5, 2.5, 3.0

plot_histograms(PREVALENCE, SEPARATION, CUTOFF)
rows = curve_data(PREVALENCE, SEPARATION)
plot_tradeoff(rows, CUTOFF)

## 3 · Change the population (and optionally the test)
Keep the cutoff fixed. Change `PREVALENCE` from `0.05` to `0.01` and then `0.20`.

**Questions:** Which curve moves most? Why? Does the cutoff that seems sensible change if the condition becomes rarer?

**Optional challenge:** Change `SEPARATION` from `2.0` to `3.0`. What happens when a test can distinguish the groups better?